# VVC_RL — training mini sau preflight đã PASS
    **Hai input:** video BDD100K `.mov` gốc + `VVC_RL_PREFLIGHT_KAGGLE_SAFE.zip` đã được kiểm tra.
    Bật Internet và GPU T4 ×2. Save & Run All để setup rồi training trực tiếp, không encode lại preflight.
    16 Train × 5 bitrate, 4 Validation × 3 bitrate, 256 frame/clip. Test30 chưa sử dụng.
    Đây là training mini mới; checkpoint cơ học của preflight không được dùng như model đã train.
    Sau này resume: đổi `TRAIN_MODE='resume'`, thay gói preflight bằng SAFE training mới nhất (đã mang theo bằng chứng preflight).


In [ ]:
from pathlib import Path
import os,sys,time,json,subprocess,threading,hashlib,zipfile,io,shutil,importlib
SESSION_STARTED=time.monotonic()
SOURCE_COMMIT='4d185279e0ba6596ff31bbf6359ab88a33f36288'
EXPECTED_SOURCE_SHA256='b0e9450f7350f1763d03232b09459b67c4259da015620f5827087b73c2363714'
REVIEWED_PREFLIGHT_SHA256='c42259b08793457186bf4b40079e121efed8b6ba63bc08f94d567afb4192062f'
REPOSITORY='https://github.com/youngan2810/VVC_RL.git'
REPO=Path('/kaggle/working/VVC_RL')  # Keep build paths identical to reviewed preflight
TOOLS=Path('/kaggle/working/vvc_rl_tools')
INPUT=Path('/kaggle/input')
PREFLIGHT=Path('/kaggle/working/vvc_rl_preflight_import')
TRAIN=Path('/kaggle/working/vvc_rl_mini')
TRAIN_MODE='new'  # new | resume
TOTAL_UPDATES_TARGET=40  # Total target, including updates already saved when resuming
VAL_EVERY_UPDATES=5
FRAMES=256
SESSION_HOURS=11.5  # Includes setup; soft stop before the session limit
assert TRAIN_MODE in ('new','resume') and TOTAL_UPDATES_TARGET>=1
assert FRAMES==256 and 0<SESSION_HOURS<=11.75
os.environ['YOLOv5_AUTOINSTALL']='false'

def live(command,cwd=None):
    print('$',*[str(v) for v in command],flush=True)
    process=subprocess.Popen(list(map(str,command)),cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    last=[time.monotonic()]
    def heartbeat():
        while process.poll() is None:
            time.sleep(15)
            if process.poll() is None and time.monotonic()-last[0]>30:print('RUNNING — elapsed',round((time.monotonic()-SESSION_STARTED)/60,1),'min',flush=True)
    threading.Thread(target=heartbeat,daemon=True).start()
    try:
        for line in process.stdout:last[0]=time.monotonic();print(line,end='',flush=True)
        if process.wait():raise RuntimeError('Command failed; keep logs and SAFE export. Do not remove assertions.')
    finally:
        if process.poll() is None:process.terminate();process.wait(timeout=30)


In [ ]:
def verify_checkout(repo,ref='HEAD'):
    repo=Path(repo).resolve()
    raw=subprocess.check_output(['git','-C',str(repo),'ls-tree','-rz','--full-tree',ref])
    failures=[];count=0
    for record in raw.split(b'\0'):
        if not record:continue
        header,name=record.split(b'\t',1);mode,kind,want=header.split();name=name.decode('utf-8');p=(repo/name).resolve()
        if kind!=b'blob' or not p.is_relative_to(repo) or not p.is_file():failures.append(name);continue
        content=p.read_bytes();actual=hashlib.sha1(b'blob '+str(len(content)).encode()+b'\0'+content).hexdigest().encode()
        if actual!=want:failures.append(name)
        count+=1
    if failures or not count:raise ValueError('Tracked source bytes differ from '+ref+': '+repr(failures[:20]))
    return count


import torch
print('PYTORCH',torch.__version__,'CUDA',torch.version.cuda,'GPU_COUNT',torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count()>=2,'Enable two GPUs'
if not REPO.exists():live(['git','clone',REPOSITORY,REPO])
print('TRACKED_RAW_BYTES_PASS',verify_checkout(REPO))
live(['git','-C',REPO,'fetch','--no-tags','origin',SOURCE_COMMIT])
live(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
verify_checkout(REPO)
sys.path.insert(0,str(REPO))
import vvcrl.config as _cfg
importlib.reload(_cfg)
assert _cfg.fingerprint(REPO)==EXPECTED_SOURCE_SHA256,'Source differs from the approved preflight'
print('EXACT_APPROVED_TRAINING_SOURCE_PASS',SOURCE_COMMIT)


In [ ]:
from vvcrl.artifacts import import_run,export_run
from vvcrl.config import digest
def input_payloads():
    found={}
    for p in sorted(INPUT.rglob('*')):
        if not p.is_file():continue
        if p.name=='VVC_RL_payload.bin':raw=p.read_bytes()
        elif p.suffix.lower()=='.zip':
            try:
                with zipfile.ZipFile(p) as z:
                    if 'VVC_RL_payload.bin' not in z.namelist():continue
                    raw=z.read('VVC_RL_payload.bin')
                    assert hashlib.sha256(raw).hexdigest()==z.read('payload_sha256.txt').decode().strip(),'Outer SAFE hash mismatch'
            except zipfile.BadZipFile:continue
        else:continue
        sha=hashlib.sha256(raw).hexdigest();found.setdefault(sha,(p,raw))
    return found
if TRAIN_MODE=='new':
    assert not (TRAIN/'checkpoints/latest.pt').exists(),'Training already exists; use resume mode'
    payloads=input_payloads()
    assert REVIEWED_PREFLIGHT_SHA256 in payloads,'Add the reviewed VVC_RL_PREFLIGHT_KAGGLE_SAFE.zip, preserving its inner .bin file'
    selected,raw=payloads[REVIEWED_PREFLIGHT_SHA256]
    TRAIN.mkdir(parents=True,exist_ok=True)
    carried=TRAIN/'reviewed_preflight_payload.bin'
    if carried.exists():assert digest(carried)==REVIEWED_PREFLIGHT_SHA256,'Existing proof differs'
    else:carried.write_bytes(raw)
    print('REVIEWED_PREFLIGHT_INPUT',selected)
else:
    if not (TRAIN/'checkpoints/latest.pt').exists():
        candidates=[]
        for sha,(p,raw) in input_payloads().items():
            with zipfile.ZipFile(io.BytesIO(raw)) as z:
                if not {'PROGRESS.json','checkpoints/latest.pt','reviewed_preflight_payload.bin'}.issubset(z.namelist()):continue
                progress=json.loads(z.read('PROGRESS.json'))
                candidates.append((int(progress['iteration']),sha,p))
        assert candidates,'Add the latest VVC_RL_MINI_TRAIN_KAGGLE_SAFE.zip for resume'
        candidates.sort(reverse=True);best_iteration=candidates[0][0]
        assert sum(x[0]==best_iteration for x in candidates)==1,'Ambiguous resume packages at the same iteration; keep only the intended latest input'
        _,_,selected=candidates[0];import_run(selected,TRAIN)
    carried=TRAIN/'reviewed_preflight_payload.bin'
assert digest(carried)==REVIEWED_PREFLIGHT_SHA256,'Carried preflight proof is not the reviewed payload'
if not (PREFLIGHT/'PREFLIGHT_GATE.json').exists():import_run(carried,PREFLIGHT)
original_gate=json.loads((PREFLIGHT/'PREFLIGHT_GATE.json').read_text())
assert original_gate['status']=='PASS_READY_FOR_MINI'
assert original_gate['binding']['source_sha256']==EXPECTED_SOURCE_SHA256
print('PREFLIGHT_PROOF_RESTORED — no preflight encoding rerun')


In [ ]:
# Only dataset identity checks + CPU contracts, not a new rollout/preflight.
from vvcrl.locked.data import VideoCatalog
catalog=VideoCatalog(INPUT,mode='mini',seed=20261111)
catalog.check_available(('train','val'))
for split in ('train','val'):
    for video in catalog.selected[split]:catalog.source(video)
print('MINI_DATASET_PASS',len(catalog.selected['train']),'Train',len(catalog.selected['val']),'Validation','Test unused')
live([sys.executable,'-m','unittest','discover','-s','tests','-q'],REPO)
live([sys.executable,'-u',REPO/'scripts/setup_kaggle.py','--tools',TOOLS,'--jobs','2'],REPO)


In [ ]:
from vvcrl.gates import binding,require_gate
from vvcrl.config import Profile,atomic_json
toolchain=json.loads((TOOLS/'TOOLCHAIN.json').read_text())
expected=binding(REPO,toolchain,Profile(frames=FRAMES),2)
differences={k:{'reviewed':original_gate['binding'].get(k),'current':expected.get(k)}
             for k in expected if original_gate['binding'].get(k)!=expected.get(k)}
atomic_json(TRAIN/'PREFLIGHT_REUSE_CHECK.json',{'compatible':not differences,'differences':differences,'source_commit':SOURCE_COMMIT,'proof_payload_sha256':REVIEWED_PREFLIGHT_SHA256})
if differences:
    print(json.dumps(differences,indent=2))
    export_run(TRAIN,TRAIN.parent/'VVC_RL_MINI_TRAIN_KAGGLE_SAFE.zip')
    raise RuntimeError('Reviewed preflight cannot be reused with a different source/profile/binary/environment. No training or automatic preflight rerun was started; keep the diagnostic export.')
require_gate(PREFLIGHT,expected)  # Verifies all stored evidence, not just PASS flags
print('VERIFIED_PREFLIGHT_REUSED_PASS — ready to train directly')


In [ ]:
progress=json.loads((TRAIN/'PROGRESS.json').read_text()) if (TRAIN/'PROGRESS.json').exists() else {'iteration':0,'episodes':0}
completed=int(progress['iteration']);additional=max(0,TOTAL_UPDATES_TARGET-completed)
if TRAIN_MODE=='new':assert completed==0,'Already trained; use resume mode'
print('MINI_TRAINING_PLAN','completed updates',completed,'additional',additional,'target',TOTAL_UPDATES_TARGET,
      '16 Train x 5 rates','4 Validation x 3 rates','256 frames','2 GPU workers')
remaining=SESSION_HOURS-(time.monotonic()-SESSION_STARTED)/3600
assert remaining>.5,'Insufficient session time after setup; preserve exports and resume in a new session'
if additional:
    command=[sys.executable,'-u','-m','vvcrl','train-mini','--output',TRAIN,'--toolchain',TOOLS/'TOOLCHAIN.json','--input',INPUT,'--tools',TOOLS,
             '--preflight',PREFLIGHT,'--frames',FRAMES,'--updates',additional,'--val-every',VAL_EVERY_UPDATES,'--hours',remaining,'--start-mini-training']
    if TRAIN_MODE=='resume':command+=['--resume',TRAIN/'checkpoints/latest.pt']
    live(command,REPO)
else:print('TARGET_ALREADY_COMPLETED — no extra training')


In [ ]:
export_run(TRAIN,TRAIN.parent/'VVC_RL_MINI_TRAIN_KAGGLE_SAFE.zip')
if (TRAIN/'PROGRESS.json').exists():print('FINAL_PROGRESS',json.loads((TRAIN/'PROGRESS.json').read_text()))
print('DOWNLOAD /kaggle/working/VVC_RL_MINI_TRAIN_KAGGLE_SAFE.zip')
print('Send the complete SAFE package for learning/validation review. Test30 was not used.')
